# Episode 15: Text diffusion: a GPT that writes all at once

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep015-diffusion.ipynb)

We turn our GPT into a masked diffusion model, which writes a story by unhiding it, train it on the laptop and on four rented GPUs, start one from our next-token models' weights, and measure it against next-token prediction, comparing our code with LLaDA's, which trains this recipe at 8 billion parameters.

[The video](https://youtu.be/LVWHstzgqE4) · [The episode's page](https://mvgpt.nenuadrian.com/ep015-diffusion/) · [The docs](https://nenuadrian.github.io/mvGPT/diffusion/) · [The code at `ep015/end`](https://github.com/nenuadrian/mvGPT/tree/ep015/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep015/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain-rope-bpe", "pretrain-ddp")

## The data, and a story with some of its tokens hidden

In [ ]:
!python data.py --bpe  # TinyStories, the BPE merges, and the stories as BPE token ids
!python diffusion.py mask

## The tests: the loss, and unhiding

In [ ]:
!python -m pytest tests/test_diffusion.py -v

## Pretraining a diffusion model
Ten minutes, to `checkpoints/pretrain-rope-bpe-diffusion`.

In [ ]:
!python pretrain.py 10 --rope --bpe --diffusion

## Writing by unhiding
With 8, 16, 32 and 64 steps, then filling in the middle of a story.

In [ ]:
!python diffusion.py write checkpoints/pretrain-rope-bpe-diffusion/model.pt --ladder 8 16 32 64
!python diffusion.py fill checkpoints/pretrain-rope-bpe-diffusion/model.pt "Once upon a time, there was a little dog named Max." "Max was never scared of the dark again." -n 2

## Against a next-token model
Bits per character (for diffusion, a bound), then stories from each, judged by the
91-million-parameter model from episode 12.

In [ ]:
!python bits.py checkpoints/pretrain-rope-bpe/model.pt checkpoints/pretrain-rope-bpe-diffusion/model.pt
!python diffusion_bench.py checkpoints/pretrain-rope-bpe-diffusion/model.pt --against checkpoints/pretrain-rope-bpe/model.pt --judge checkpoints/pretrain-ddp/model.pt

## From a next-token model
Five minutes, starting from episode 11's model.

In [ ]:
!python pretrain.py 5 --rope --bpe --diffusion --init checkpoints/pretrain-rope-bpe/model.pt
!python bits.py checkpoints/pretrain-rope-bpe-diffusion-converted/model.pt

## On four GPUs
The episode trained both on four rented RTX 4090s, with `pretrain_ddp.py --diffusion`: 30
minutes from scratch, and 15 from episode 12's model. Here are those two models.

In [ ]:
fetch("diffusion-ddp", "diffusion-ddp-converted")

In [ ]:
!python bits.py checkpoints/pretrain-ddp/model.pt checkpoints/diffusion-ddp/model.pt checkpoints/diffusion-ddp-converted/model.pt
!python diffusion_bench.py checkpoints/diffusion-ddp-converted/model.pt --against checkpoints/pretrain-ddp/model.pt --judge checkpoints/pretrain-ddp/model.pt --stories 8
!python diffusion.py write checkpoints/diffusion-ddp/model.pt --ladder 8 16 32 64
!python diffusion.py fill checkpoints/diffusion-ddp-converted/model.pt "Once upon a time, there was a little dog named Max." "Max was never scared of the dark again." -n 2

Next: [episode 16, GRPO: reinforcement learning without a critic](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep016-grpo.ipynb)